# Suite ALG — The predicate algebra

`Predicates` extends mbse-expressions' Basic with terms about a store's data: quantifiers over a schema's objects
(`forall`, `exists`, `count` over an `extent`), `linked`, and weighted choices (`choice` of `option`s). Mandatory,
possible and forbidden links are predicates of the algebra.

In [ ]:
import { Expressions as E } from "@mbse/expressions";
import { Terms } from "@mbse/expressions/Framework";
import { Constraints as C, Predicates as P, Queries as Q, Validators as V } from "@mbse/patterns";
import { Errors, JSON as SchemaJSON, Proxies, Schemas as S } from "@mbse/schemas/Framework";
import { Contact, Phone, assert, book, equal, listed, raises, same } from "./support.js";

const [c, p, d, x] = [E.variable("c"), E.variable("p"), E.variable("d"), E.variable("x")];
const owns = P.exists("p", Phone, P.linked(c, "phones", p));
const mandatory = P.forall("c", Contact, owns); // every contact has a phone
const forbidden = P.forall("c", Contact, owns.not_()); // no contact has one
const possible = P.forall("c", Contact, P.choice([0.35, owns], [0.65, owns.not_()])); // 35% of contacts have one

## ALG-01 · The algebra's terms are written and checked

In [ ]:
{
  assert(mandatory.data instanceof P.OfForall && mandatory.data.name === "c" && mandatory.data.collection.schema === "Contact");
  assert(P.extent("Contact").data.schema === "Contact" && P.extent(Contact).data.schema === "Contact"); // a schema, or its name
  assert(P.DIALECT.base === E.DIALECT && P.DIALECT.validate(mandatory.data, { core: true }).length === 0);
  assert(same([...P.DIALECT.kinds().keys()].slice(-7), ["extent", "forall", "exists", "count", "linked", "choice", "option"]));
  assert(same([...P.DIALECT.schemas.keys()].filter((n) => n.startsWith("Patterns.")), ["Patterns.OfExtent", "Patterns.OfForall",
    "Patterns.OfExists", "Patterns.OfCount", "Patterns.OfLinked", "Patterns.OfChoice", "Patterns.OfOption"]));
  assert(E.DIALECT.validate(mandatory.data)[0]!.startsWith("not an expression: ")); // Basic alone cannot check it
  const mixed = c.has("name").and_(owns); // Basic's writers hold the algebra's terms, and its writers Basic's
  assert(P.DIALECT.validate(mixed.data, { bound: ["c"], core: true }).length === 0 && owns.not_().data instanceof E.OfOperation.Data);
  assert(same(P.DIALECT.validate(P.linked(c, "phones", p).data), ["source: variable 'c' is not bound",
    "target: variable 'p' is not bound"]));
  assert(same(P.DIALECT.validate(P.choice([0.5, true], [0.4, false]).data), ["a choice's weights must sum to 1, got 0.9"]));
  assert(same(P.DIALECT.validate(P.choice([-1.0, true], [2.0, false]).data), [
    "argument 0: an option's weight must be positive, got -1.0"]));
  assert(same(P.DIALECT.validate(P.choice().data), ["a choice needs an option"]));
  assert(same(P.DIALECT.validate(new P.OfChoice([E.literal(true).data])), ["a choice's arguments are options"]));
  raises(TypeError, () => P.extent(new S.OfObject.Builder().create()), "expected a named schema or its name, got ");
}

## ALG-02 · The algebra is evaluated over the store

In [ ]:
{
  const store = book();
  const ann = store.Contact().name("ann").age(30n).phones((e: any) => e.phone((q: any) => q.number("1"))).create();
  const bob = store.Contact().name("bob").create();
  listed(store, ann, bob);
  const evaluate = new P.Evaluator(store);
  assert(evaluate.run(mandatory) === false && evaluate.run(forbidden) === false && evaluate.run(possible) === true); // bob has none
  assert(evaluate.run(P.forall("c", Contact, owns.or_(c.has("age").not_()))) === true); // those without an age may have none
  assert(evaluate.run(P.count("c", Contact, c.has("age"))) === 1n && same(evaluate.run(P.extent(Contact)) as unknown[], [ann, bob]));
  assert(evaluate.run(P.exists("c", Contact, c.name.eq("bob").and_(owns.not_()))) === true);
  assert(evaluate.run(P.choice([0.5, x], [0.5, false]), { x: null }) === null); // Kleene's disjunction
  assert(evaluate.run(P.choice([0.5, x], [0.5, true]), { x: null }) === true);
  assert(evaluate.run(P.choice([0.5, false], [0.5, false])) === false);
  assert(evaluate.run(P.linked(c, "phones", p), { c: ann, p: null }) === null); // an unknown end: unknown
  listed(store, store.Contact().name("cy").create());
  assert((evaluate.run(P.extent(Contact)) as unknown[]).length === 2 && (new P.Evaluator(store).run(P.extent(Contact)) as unknown[]).length === 3);
  raises(TypeError, () => evaluate.run(P.linked(c, "phones", p), { c: 1n, p: ann }), "linked expects an object as its source, got int");
  raises(TypeError, () => evaluate.run(P.linked(c, "phones", p), { c: ann, p: "x" }), "linked expects an object as its target, got str");
  raises(TypeError, () => evaluate.run(P.linked(c, "pagers", p), { c: ann, p: ann }), "'Contact' has no adjacency 'pagers'");
  raises(Errors.AttributeError, () => evaluate.run(P.extent("Nope")), "no schema registered as 'Nope'");
  raises(TypeError, () => evaluate.run(P.choice([1.0, 3n])), "a choice's options must be bools, got int");
  // A relation of more than two links: linked names the link.
  const Assigned = new S.OfRelation.Builder().name("Assigned").links("desk", "person", "seat").create();
  const Person = new S.OfObject.Builder().name("Person").ref().relations((r) => r.name("desks").of(Assigned).me("person")).create();
  const Seat = new S.OfObject.Builder().name("Seat").ref().relations((r) => r.name("desks").of(Assigned).me("seat")).create();
  const Desk = new S.OfObject.Builder().name("Desk").ref().singleton("office.Desk").relations(
    (r) => r.name("assigned").of(Assigned).me("desk")).create();
  const office: any = new Proxies.OfStore();
  for (const schema of [Assigned, Person, Seat, Desk]) office.register(schema);
  const [desk, pam, seat] = [office.singleton("office.Desk"), office.Person().create(), office.Seat().create()];
  office.Desk(desk).assigned((e: any) => e.person(pam).seat(seat)).update();
  const at = new P.Evaluator(office);
  assert(at.run(P.linked(d, "assigned", x, "person"), { d: desk, x: pam }) === true);
  assert(at.run(P.linked(d, "assigned", x, "seat"), { d: desk, x: seat }) === true);
  assert(at.run(P.linked(d, "assigned", x, "seat"), { d: desk, x: pam }) === false);
  raises(Errors.ValueError, () => at.run(P.linked(d, "assigned", x), { d: desk, x: pam }),
    "linked needs a link for 'assigned', whose relation has links 'person', 'seat'");
  const seated = new C.OfPredicate.Builder().name("Seated").symbols({ d: Desk, x: Person }).rule(P.linked(d, "assigned", x)).create();
  assert(same(new Q.Scan(office).explain(seated), ["d: scan Desk", "x: scan Person, then 1 test"])); // which link: no hop
  const named = new C.OfPredicate.Builder(seated).rule(P.linked(d, "assigned", x, "person")).clone();
  assert(same(new Q.Scan(office).explain(named), ["d: scan Desk", "x: d.assigned to person, any number, then 1 test"]));
  assert(equal([...new Q.Scan(office).select(named)], [{ d: desk, x: pam }]));
}

## ALG-03 · Predicates written in the algebra: about the whole store, through links, and as data

In [ ]:
{
  const store = book();
  const ann = store.Contact().name("ann").age(30n).phones((e: any) => e.phone((q: any) => q.number("1"))).create();
  const bob = store.Contact().name("bob").create();
  listed(store, ann, bob);
  const EveryoneHasAPhone = new C.OfPredicate.Builder().name("EveryoneHasAPhone").rule(mandatory).create(); // no symbols
  assert(same(V.Validate(store, [EveryoneHasAPhone])(Contact, ann), ["the store: 'EveryoneHasAPhone' does not hold"]));
  const query = new Q.Scan(store);
  assert([...query.select(EveryoneHasAPhone)].length === 0 && same(query.explain(EveryoneHasAPhone), ["first 1 test"]));
  store.Contact(bob).phones((e: any) => e.phone((q: any) => q.number("2"))).update();
  assert(equal([...query.select(EveryoneHasAPhone)], [{}])); // one match, empty, when it holds
  const Numbered = new C.OfPredicate.Builder().name("Numbered").symbols({ c: Contact, p: Phone }).rule(
    P.linked(c, "phones", p).and_(p.has("number"))).create();
  assert(same(query.explain(Numbered), ["c: scan Contact", "p: c.phones to phone, any number, then 2 tests"]));
  assert(equal([...query.select(Numbered)].map((m: any) => [m.c.name, m.p.number]), [["ann", "1"], ["bob", "2"]]));
  const Owner = new C.OfPredicate.Builder().name("Owner").symbols({ c: Contact, p: Phone }).rule(P.linked(p, "owners", c)).create();
  assert(same(query.explain(Owner), ["p: scan Phone", "c: p.owners to owner, at most one, then 1 test"]));
  assert(same(query.explain(new C.OfPredicate.Builder(Owner).rule(P.linked(p, "owners", c, "phone")).clone()), [
    "c: scan Contact", "p: scan Phone, then 1 test"])); // its own link: not a hop
  assert(same(query.explain(new C.OfPredicate.Builder(Owner).rule(P.linked(c, "nope", p)).clone()), [
    "c: scan Contact", "p: scan Phone, then 1 test"])); // no such adjacency
  assert(same(query.explain(new C.OfPredicate.Builder(Owner).rule(P.linked(c, "phones", c)).clone()), [
    "c: scan Contact, then 1 test", "p: scan Phone"])); // back to itself
  const predicates = new C.OfSet.Builder().predicates(EveryoneHasAPhone, Numbered, (b) => b.name("Possible").rule(possible)).create();
  const schemas = new C.OfStore(store);
  const text = SchemaJSON.ToJSON(schemas).Reachable(C.OfSet.Schema, predicates);
  assert(text.includes('"$schema": "Patterns.OfForall"') && text.includes('"$schema": "Patterns.OfOption"'));
  const copy = SchemaJSON.FromJSON(schemas).Reachable(C.OfSet.Schema, text) as C.OfSet.Data;
  assert(copy.predicates.every((a, i) => Terms.same(a.rule, predicates.predicates[i]!.rule)));
  assert((copy.predicates[2]!.rule as any).body.options[0].weight === 0.35);
}